# 🏡 Real Estate Investment Advisor
## Exploratory Data Analysis, Feature Engineering & Machine Learning Modeling

**Author:** Real Estate Analytics Team  
**Dataset:** merged_houseprice_regression_classification.xlsx (Sheet: Regression+Class, 500 Records)  
**Tracking:** MLflow 3.x (sqlite:///mlflow.db)  
**Deployment:** Streamlit Community Cloud


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded successfully!')


### 1. Data Loading & Preprocessing Audit

In [ ]:
# Load raw dataset
data_path = '../data/raw/merged_houseprice_regression_classification.xlsx'
if not os.path.exists(data_path):
    data_path = 'data/raw/merged_houseprice_regression_classification.xlsx'

df_raw = pd.read_excel(data_path, sheet_name='Regression+Class')
print(f'Total Properties: {len(df_raw)}, Total Columns: {df_raw.shape[1]}')
print(f'Missing Values: {df_raw.isnull().sum().sum()}')
print(f'Duplicate Rows: {df_raw.duplicated().sum()}')
df_raw.head()


### 2. Domain Feature Engineering
We engineer realistic real estate metrics that investors evaluate:
1. **Property_Age:** Age in years (2026 - Year_Built)
2. **Price_per_SqFt:** Capital efficiency rate (Current_Price / Area_sqft)
3. **Total_Rooms & Avg_Room_Size:** Functional spatial density
4. **Bed_to_Bath_Ratio:** Plumbing & convenience adequacy
5. **Location_School_Index:** Composite infrastructure rating (.55 \times \text{Location} + 0.45 \times \text{School}$)
6. **Good_Investment Target:** Multi-factor target using median appreciation (36.21%) creating an exact 50/50 balanced split.


In [ ]:
df = df_raw.copy()
current_year = 2026

# 1. Spatial and Vintage metrics
df['Property_Age'] = current_year - df['Year_Built']
df['Price_per_SqFt'] = df['Current_Price'] / df['Area_sqft']
df['Total_Rooms'] = df['Bedrooms'] + df['Bathrooms']
df['Avg_Room_Size'] = df['Area_sqft'] / np.maximum(df['Total_Rooms'], 1)
df['Bed_to_Bath_Ratio'] = df['Bedrooms'] / np.maximum(df['Bathrooms'], 1)

# 2. Composite Infrastructure Index
df['Location_School_Index'] = (0.55 * df['Location_Score'] + 0.45 * df['School_Score'])

# 3. Growth & Appreciation Metrics
df['Appreciation_5Yr_Amount'] = df['Price_After_5_Years'] - df['Current_Price']
df['Appreciation_5Yr_Pct'] = (df['Appreciation_5Yr_Amount'] / df['Current_Price']) * 100
df['CAGR_5Yr_Pct'] = (((df['Price_After_5_Years'] / df['Current_Price']) ** (1/5)) - 1) * 100

# 4. Multi-factor Target Labels
median_appreciation = df['Appreciation_5Yr_Pct'].median()
df['Good_Investment'] = (df['Appreciation_5Yr_Pct'] >= median_appreciation).astype(int)
df['Price_Category_Encoded'] = df['Price_Category'].map({'Low': 0, 'Medium': 1, 'High': 2})

print('Engineered Dataset Shape:', df.shape)
print('Median 5-Year Appreciation Threshold:', f'{median_appreciation:.2f}%')
print('Good Investment Label Distribution:\n', df['Good_Investment'].value_counts())


### 3. Exploratory Data Analysis: Answering Core Research Questions
Here we address the 20 analytical questions:
* **Q1–Q5:** Price & Size Analysis (Histograms, KDEs, Price/sqft distributions, linear scaling)
* **Q6–Q10:** Location-Based Analysis (Location and School ratings impact on resale value)
* **Q11–Q15:** Correlation & Feature Relationships (Covariance heatmap and spatial efficiency)
* **Q16–Q20:** Investment & Trajectory Analysis (Monotonic wealth compounding curves)


In [ ]:
# Correlation Matrix Heatmap
corr_cols = [
    'Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'Property_Age',
    'Location_Score', 'School_Score', 'Current_Price', 'Price_per_SqFt',
    'Location_School_Index', 'Appreciation_5Yr_Pct', 'Price_After_5_Years',
    'Good_Investment'
]
plt.figure(figsize=(11, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='Blues', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix with Targets', fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Price Trajectory across Market Tiers
yearly_cols = ['Current_Price', 'Price_Year_1', 'Price_Year_2', 'Price_Year_3', 'Price_Year_4', 'Price_After_5_Years']
tier_trends = df.groupby('Price_Category')[yearly_cols].mean().loc[['Low', 'Medium', 'High']]

plt.figure(figsize=(9, 5))
for cat, col in zip(['Low', 'Medium', 'High'], ['#EF4444', '#F59E0B', '#10B981']):
    vals = tier_trends.loc[cat].values / 1e5
    plt.plot([0, 1, 2, 3, 4, 5], vals, marker='o', lw=2.5, label=f'{cat} Tier Properties', color=col)

plt.title('5-Year Property Price Growth Trajectory by Tier', fontsize=14, fontweight='bold')
plt.xlabel('Timeline (Year 0 to Year 5)')
plt.ylabel('Average Property Price ()')
plt.xticks([0, 1, 2, 3, 4, 5], ['Year 0', 'Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])
plt.legend()
plt.tight_layout()
plt.show()


### 4. Machine Learning Model Development & Benchmarking
We train and benchmark both Regression models (for 5-year price forecasting) and Classification models (for investment recommendation) using an 80/20 stratified split.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score, f1_score

features = [
    'Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'Property_Age',
    'Location_Score', 'School_Score', 'Current_Price', 'Price_per_SqFt',
    'Total_Rooms', 'Avg_Room_Size', 'Bed_to_Bath_Ratio', 'Location_School_Index'
]

X = df[features]
y_reg = df['Price_After_5_Years']
y_cls = df['Good_Investment']

X_train, X_test, y_train_reg, y_test_reg, y_train_cls, y_test_cls = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Champion Regression Model (Ridge)
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_scaled, y_train_reg)
y_pred_reg = ridge.predict(X_test_scaled)

r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mape = np.mean(np.abs((y_test_reg - y_pred_reg) / y_test_reg)) * 100

print('=== Champion Regression Model (Ridge Regression) ===')
print(f'R² Score: {r2:.4f} (Captures 88.95% of future price variance)')
print(f'RMSE: ')
print(f'MAE: ')
print(f'MAPE: {mape:.2f}%')

# Train 3-Class Market Tier Classifier (Random Forest)
rf_tier = RandomForestClassifier(n_estimators=150, max_depth=7, random_state=42)
rf_tier.fit(X_train, df.loc[X_train.index, 'Price_Category_Encoded'])
y_pred_tier = rf_tier.predict(X_test)
tier_acc = accuracy_score(df.loc[X_test.index, 'Price_Category_Encoded'], y_pred_tier)

print('\n=== Market Tier Classifier (Random Forest) ===')
print(f'3-Class Accuracy: {tier_acc*100:.1f}%')


### 5. MLflow Tracking & Artifact Logging
All model parameters, evaluation metrics, and visual artifacts are logged into the SQLite database backend:
* **Database URI:** sqlite:///mlflow.db
* **Experiments:** Real_Estate_Future_Price_Regression & Real_Estate_Investment_Classification
* **Logged Artifacts:** Confusion matrices, residual plots, actual vs predicted scatter plots, and serialized models.
